# 03a — OLS (naiv baseline)

Estimerer basismodellen med OLS og HAC-standardfeil. Brukes som naiv baseline for sammenligning med 2SLS-hovedmodellen i `03d_2sls.ipynb` — forskjellen i koeffisienten på `cons_NO4` illustrerer omfanget av endogenitetsskjevheten påvist i `02a_diagnostikk.ipynb`.

**Input:** `intermediate/df_iso.parquet`

**Output:** `intermediate/preds_ols.parquet`, `intermediate/models_ols.pkl`

In [12]:
import pandas as pd
import statsmodels.api as sm

from src.config import (
    INTERMEDIATE_DIR, TARGET, OLS_FEATURES, TRAIN_YEARS, TEST_YEARS, apply_style
)
from src.evaluation import eval_metrics
from src.model_training import (
    load_prepared_iso_data, split_features_target, make_prediction_frame,
    save_model_artifacts,
)

apply_style()

In [13]:
df_iso = load_prepared_iso_data(INTERMEDIATE_DIR)
X_train, y_train, X_test, y_test, train_mask, test_mask = split_features_target(
    df_iso,
    feature_cols=OLS_FEATURES,
    target=TARGET,
    train_years=TRAIN_YEARS,
    test_years=TEST_YEARS,
)

print(f"Trening: {len(X_train):,} ISO-timer")
print(f"Test:    {len(X_test):,} ISO-timer")
print(f"Features: {len(OLS_FEATURES)}")

Trening: 25,072 ISO-timer
Test:    9,844 ISO-timer
Features: 37


In [14]:
X_train_c = sm.add_constant(X_train)
X_test_c = sm.add_constant(X_test, has_constant="add")

ols_model = sm.OLS(y_train, X_train_c).fit(
    cov_type="HAC",
    cov_kwds={"maxlags": 24},
)

print(ols_model.summary())

                            OLS Regression Results                            
Dep. Variable:              price_NO4   R-squared:                       0.268
Model:                            OLS   Adj. R-squared:                  0.267
Method:                 Least Squares   F-statistic:                     10.31
Date:                Fri, 29 May 2026   Prob (F-statistic):           2.72e-58
Time:                        15:18:28   Log-Likelihood:            -1.7495e+05
No. Observations:               25072   AIC:                         3.500e+05
Df Residuals:                   25034   BIC:                         3.503e+05
Df Model:                          37                                         
Covariance Type:                  HAC                                         
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
const          -357.8472     92.514     -3.868

In [15]:
preds = make_prediction_frame(
    df=df_iso,
    mask=test_mask,
    actual=y_test,
    prediction_col="OLS",
    predictions=ols_model.predict(X_test_c),
)
metrics = eval_metrics(preds["actual"], preds["OLS"])
display(pd.DataFrame([metrics], index=["OLS"]))

,MAE,RMSE,R²,N
OLS,154.6,222.8,0.2425,9844


In [16]:
payload = {
    "model_name": "OLS",
    "prediction_col": "OLS",
    "feature_cols": OLS_FEATURES,
    "target_col": TARGET,
    "train_years": TRAIN_YEARS,
    "test_years": TEST_YEARS,
    "estimator": ols_model,
    "metrics": metrics,
    "diagnostics": {
        "beta_cons_NO4": float(ols_model.params["cons_NO4"]),
        "se_cons_NO4":  float(ols_model.bse["cons_NO4"]),
    },
}

save_model_artifacts("ols", preds, payload, intermediate_dir=INTERMEDIATE_DIR)
print(f"Lagret {INTERMEDIATE_DIR}preds_ols.parquet")
print(f"Lagret {INTERMEDIATE_DIR}models_ols.pkl")

Lagret intermediate/preds_ols.parquet
Lagret intermediate/models_ols.pkl


---
## Sesongspesifikk OLS

Én OLS per sesong (Vinter/Vår/Sommer/Høst) for å fange opp at marginal-
effekten av lokal etterspørsel på prisen kan variere med tilbudssiden
(hydrologi, vindprofil, fyllingsgrad). Brukes som komplement til den
globale OLS-modellen i notebook 05 og 06.

Månedsdummiene er fjernet fra featuresettet siden hver sesong dekker tre
måneder — de ville vært nær-kolineære med en sesongspesifikk konstant.

**Output:** `intermediate/models_ols_seasonal.pkl`

In [17]:
import pickle

from src.config import BASE_FEATURES, HOUR_DUMMIES, SEASON_ORDER, STARGATE_MW, STARGATE_TOTAL_MW

# Per-sesong featuresett — dropp månedsdummier (nær-kolinere innen en sesong)
SEASONAL_FEATURES = BASE_FEATURES + HOUR_DUMMIES
SEASONS = list(SEASON_ORDER.keys())  # Winter, Spring, Summer, Autumn

seasonal_models = {}
for season in SEASONS:
    mask_s = train_mask & (df_iso["season"] == season)
    X_s = df_iso.loc[mask_s, SEASONAL_FEATURES]
    y_s = df_iso.loc[mask_s, TARGET]
    X_s_c = sm.add_constant(X_s, has_constant="add")
    seasonal_models[season] = sm.OLS(y_s, X_s_c).fit(
        cov_type="HAC", cov_kwds={"maxlags": 24}
    )
    b  = seasonal_models[season].params["cons_NO4"]
    se = seasonal_models[season].bse["cons_NO4"]
    print(f"{SEASON_ORDER[season]:<7s} N={len(X_s):>6,}  β_cons={b:+.4f}  (SE={se:.4f})  "
          f"ΔP(+{STARGATE_MW} MW)={b*STARGATE_MW:+.1f} NOK/MWh")

# Simulerte ΔP per time for begge scenarier (lineart → β_sesong × ΔMW)
betas_by_season = {s: seasonal_models[s].params["cons_NO4"] for s in SEASONS}
sim_results = {}
for delta in [STARGATE_MW, STARGATE_TOTAL_MW]:
    dp = df_iso["season"].map(betas_by_season).astype(float) * delta
    sim_results[delta] = pd.DataFrame(
        {"dP": dp.values, "season": df_iso["season"].values},
        index=df_iso.index,
    )

with open(INTERMEDIATE_DIR + "models_ols_seasonal.pkl", "wb") as f:
    pickle.dump({
        "models":      seasonal_models,
        "features":    SEASONAL_FEATURES,
        "sim_results": sim_results,
    }, f)

print(f"\nLagret {INTERMEDIATE_DIR}models_ols_seasonal.pkl")

Vinter  N= 6,385  β_cons=+0.6900  (SE=0.1162)  ΔP(+230 MW)=+158.7 NOK/MWh
Vår     N= 5,887  β_cons=+0.2672  (SE=0.0363)  ΔP(+230 MW)=+61.5 NOK/MWh
Sommer  N= 6,688  β_cons=-0.1632  (SE=0.0331)  ΔP(+230 MW)=-37.5 NOK/MWh
Høst    N= 6,112  β_cons=+0.1696  (SE=0.0627)  ΔP(+230 MW)=+39.0 NOK/MWh

Lagret intermediate/models_ols_seasonal.pkl
